<a href="https://colab.research.google.com/github/shiragelb/NCC-Statistical-Reports/blob/main/alignment_of_tables_and_headers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os
import io

# 1️⃣ Authenticate
auth.authenticate_user()
drive_service = build('drive', 'v3')

# 2️⃣ Folder ID of the specific directory
folder_id = "1xH-6eyhl0XDe0DfS33o0xZAIdTLPkyQj"  # <-- ID of "excel_and_json_2019_2021_2024"

# 3️⃣ Recursive listing of all files in this folder
def list_all_files_in_folder_recursive(parent_id, parent_path=""):
    all_files = []
    query = f"'{parent_id}' in parents and trashed=false"
    page_token = None

    while True:
        response = drive_service.files().list(
            q=query,
            spaces='drive',
            fields='nextPageToken, files(id, name, mimeType)',
            pageToken=page_token
        ).execute()

        for item in response.get('files', []):
            item_path = f"{parent_path}/{item['name']}" if parent_path else item['name']
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                all_files.extend(list_all_files_in_folder_recursive(item['id'], item_path))
            else:
                all_files.append({
                    "file_name": item['name'],
                    "file_path": item_path,
                    "file_id": item['id']
                })

        page_token = response.get('nextPageToken', None)
        if page_token is None:
            break

    return all_files

# 4️⃣ Download all files to local directory
def download_files(files_list, download_dir="/content/excel_and_json_2019_2021_2024"):
    os.makedirs(download_dir, exist_ok=True)

    for file in files_list:
        local_path = os.path.join(download_dir, file['file_path'].replace("/", os.sep))
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        request = drive_service.files().get_media(fileId=file['file_id'])
        fh = io.FileIO(local_path, "wb")
        downloader = MediaIoBaseDownload(fh, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file['file_name']} {int(status.progress() * 100)}%")

        print(f"✅ Saved {file['file_name']} to {local_path}")

# 5️⃣ Run everything
files_list = list_all_files_in_folder_recursive(folder_id)
download_files(files_list)


⬇️ Downloading 162_1_2024.csv 100%
✅ Saved 162_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/162_1_2024.csv
⬇️ Downloading 172_1_2024.csv 100%
✅ Saved 172_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/172_1_2024.csv
⬇️ Downloading 160_1_2024.csv 100%
✅ Saved 160_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/160_1_2024.csv
⬇️ Downloading 186_1_2024.csv 100%
✅ Saved 186_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/186_1_2024.csv
⬇️ Downloading 161_1_2024.csv 100%
✅ Saved 161_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/161_1_2024.csv
⬇️ Downloading 190_1_2024.csv 100%
✅ Saved 190_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/190_1_2024.csv
⬇️ Downloading 180_1_2024.csv 100%
✅ Saved 180_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/180_1_2024.csv
⬇️ Downloading 191_1_2024.csv 100%
✅ Saved 191_1_2024.csv to /content/excel_and_json_2019_2021_2024/2024/1/191_1_2024.csv
⬇️ Downloading 176_1_202

In [5]:
import os
import pandas as pd
import json
import re

def is_hebrew(text):
    """Return True if the text contains Hebrew letters."""
    if not isinstance(text, str):
        return False
    return bool(re.search(r'[\u0590-\u05FF]', text))

def clean_text_list(text_list):
    """Remove numbers, NaNs, and non-Hebrew/English unwanted strings."""
    cleaned = []
    for t in text_list:
        if pd.isna(t):
            continue
        t = str(t).strip()
        if not t:
            continue
        if t.lower() in ['nan', 'none']:
            continue
        # Remove numeric values
        # if re.match(r'^[-+]?[0-9]*[.,]?[0-9]+$', t):
        #     continue
        # Keep only Hebrew
        if is_hebrew(t):
            cleaned.append(t)
    return cleaned

def extract_hebrew_from_csv_per_chapter(base_dir, output_dir, rows_to_extract=3):
    """
    Extract Hebrew text from CSV files per chapter.
    Saves one JSON per chapter inside output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)

    for year_folder in os.listdir(base_dir):
        year_path = os.path.join(base_dir, year_folder)
        if not os.path.isdir(year_path):
            continue

        for chapter_folder in os.listdir(year_path):
            chapter_path = os.path.join(year_path, chapter_folder)
            if not os.path.isdir(chapter_path):
                continue

            chapter_result = {}

            for file in os.listdir(chapter_path):
                if file.endswith('.csv'):
                    csv_path = os.path.join(chapter_path, file)
                    try:
                        df = pd.read_csv(csv_path, dtype=str)
                    except Exception as e:
                        print(f"Error reading {csv_path}: {e}")
                        continue

                    texts = []

                    # Include all column headers
                    if df.columns is not None:
                        texts.extend(df.columns.astype(str).tolist())

                    # Include all rows and all cells
                    for i in range(len(df)):
                        row_text = df.iloc[i].astype(str).tolist()
                        texts.extend(row_text)

                    # Clean texts
                    cleaned_texts = clean_text_list(texts)

                    # Use filename without .csv as key
                    key = file.replace('.csv', '')
                    chapter_result[key] = cleaned_texts

            # Save JSON per chapter
            chapter_json_path = os.path.join(chapter_path, f"{chapter_folder}_{year_folder}_headers.json")
            with open(chapter_json_path, 'w', encoding='utf-8') as f:
                json.dump(chapter_result, f, ensure_ascii=False, indent=2)

            print(f"Saved chapter JSON: {chapter_json_path}")

# Example usage
base_dir = "/content/excel_and_json_2019_2021_2024/"
output_dir = "/content/hebrew_per_chapter_json/"
extract_hebrew_from_csv_per_chapter(base_dir, output_dir)


Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/12/12_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/1/1_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/5/5_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/10/10_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/3/3_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/6/6_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/11/11_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/13/13_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/2/2_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/8/8_2022_headers.json
Saved chapter JSON: /content/excel_and_json_2019_2021_2024/2022/7/7_2022_headers.json
Saved chapter JSON: /content/excel_and_json_20

In [2]:
%%writefile real_embeddings.py

import numpy as np
import pickle
import os
import hashlib

try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMER_AVAILABLE = True
except:
    TRANSFORMER_AVAILABLE = False
    print("Install with: !pip install sentence-transformers")

class RealEmbeddingGenerator:
    def __init__(self, model_name="sentence-transformers/LaBSE", cache_dir="cache"):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        self.embedding_cache = {}

        if TRANSFORMER_AVAILABLE:
            self.model = SentenceTransformer(model_name)
            self.dimension = self.model.get_sentence_embedding_dimension()
        else:
            self.model = None
            self.dimension = 768

    def get_text_hash(self, text):
        return hashlib.md5(text.encode('utf-8')).hexdigest()

    def generate_embedding(self, text, use_cache=True):
        text_hash = self.get_text_hash(text)

        if use_cache and text_hash in self.embedding_cache:
            return self.embedding_cache[text_hash]

        if self.model:
            embedding = self.model.encode(text, convert_to_numpy=True)
        else:
            # Fallback to deterministic random
            np.random.seed(int(text_hash[:8], 16) % 10000)
            embedding = np.random.randn(self.dimension)

        if use_cache:
            self.embedding_cache[text_hash] = embedding

        return embedding

    def generate_batch(self, texts, show_progress=True):
        if self.model:
            return self.model.encode(texts,
                                    batch_size=32,
                                    show_progress_bar=show_progress,
                                    convert_to_numpy=True)
        else:
            return np.array([self.generate_embedding(t) for t in texts])

    def save_cache(self):
        cache_file = os.path.join(self.cache_dir, "embedding_cache.pkl")
        with open(cache_file, 'wb') as f:
            pickle.dump(self.embedding_cache, f)


Writing real_embeddings.py


In [30]:
import os
import json
import numpy as np
from real_embeddings import RealEmbeddingGenerator

def cosine_similarity(a, b):
    a = np.array(a).flatten()
    b = np.array(b).flatten()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def normalize_scores(scores):
    """Normalize a list of scalar scores to 0-1 range."""
    scores = [float(s) for s in scores]
    if not scores:
        return scores
    min_s, max_s = min(scores), max(scores)
    if max_s - min_s < 1e-8:
        return [1.0] * len(scores)
    return [(s - min_s) / (max_s - min_s) for s in scores]

def semantic_alignment_avg_word_header(table_names_json_path, headers_json_path, threshold=0.2):
    """
    Align table names to headers (features) using semantic similarity.
    Each header is represented as the average of embeddings of its words.
    """
    # Load JSONs
    with open(table_names_json_path, 'r', encoding='utf-8') as f:
        table_names = json.load(f)
    with open(headers_json_path, 'r', encoding='utf-8') as f:
        table_headers = json.load(f)

    embedder = RealEmbeddingGenerator()
    report = {}

    table_name_keys = list(table_names.keys())
    table_name_texts = [table_names[k] for k in table_name_keys]

    print("Embedding all table names...")
    table_name_embeddings = embedder.generate_batch(table_name_texts, show_progress=True)
    print(f"Embedded {len(table_name_embeddings)} table names.\n")

    for header_key, header_texts in table_headers.items():
        print(f"\n--- Processing header key: {header_key} ---")
        print(f"Header texts: {header_texts}")

        # Step 1: embed each word in each header
        header_word_embeddings = []
        for h in header_texts:
            words = h.split()  # simple tokenization by space
            if words:
                word_embs = embedder.generate_batch(words, show_progress=False)
                # Average embeddings for this header
                avg_emb = np.mean(np.array(word_embs), axis=0)
            else:
                avg_emb = np.zeros(table_name_embeddings[0].shape)
            header_word_embeddings.append(avg_emb)

        matches = []

        for i, table_emb in enumerate(table_name_embeddings):
            table_name = table_name_texts[i]
            table_emb = np.array(table_emb).flatten()
            print(f"\nComparing to table: {table_name} (key={table_name_keys[i]})")

            # Compute similarity to each header vector
            sims = [cosine_similarity(table_emb, h_emb) for h_emb in header_word_embeddings]
            sims_norm = normalize_scores(sims)
            avg_sim = float(np.mean(sims_norm))

            for h_text, s_raw, s_norm in zip(header_texts, sims, sims_norm):
                print(f"  Header: '{h_text}' | raw_sim={s_raw:.4f} | norm_sim={s_norm:.4f}")

            print(f"Average normalized similarity: {avg_sim:.4f}")

            if avg_sim >= threshold:
                matches.append({
                    "table_name_key": table_name_keys[i],
                    "table_name": table_name,
                    "similarity": avg_sim
                })

        matches = sorted(matches, key=lambda x: x['similarity'], reverse=True)
        print(f"Matches above threshold ({threshold}): {matches}")
        report[header_key] = matches

    return report

# Example usage
chapter_report = semantic_alignment_avg_word_header(
    "/content/excel_and_json_2019_2021_2024/2019/4/4_2019.json",
    "/content/excel_and_json_2019_2021_2024/2019/4/4_2019_headers.json",
    threshold=0.2
)

print(json.dumps(chapter_report, ensure_ascii=False, indent=2))

Embedding all table names...
Embedded 10 table names.


--- Processing header key: 1_4_2019 ---
Header texts: ['סידרה 1', 'גרושים', 'נשואים', 'פרודים', 'שני ההורים אינם בחיים', 'אלמן/ה', 'מצב משפחתי', 'עמודה1', 'אחר', 'שני ההורים אינם בחיים', 'אלמן/אלמנה', 'פרודים', 'נשואים', 'גרושים', 'תוויות שורה', 'מספר מטופלים', 'באחוזים', 'נשואים', 'פרודים', 'גרושים', 'אלמנים', 'שני ההורים אינם בחיים', 'לא נענה', 'סה"כ']

Comparing to table: תרשים 4א': התפלגות סוג משפחה של ילדים* בפנימיות של המינהל לחינוך התיישבותי פנימייתי ועליית הנוער / תשע"ח (2017/18) (key=1_4_2019)
  Header: 'סידרה 1' | raw_sim=-0.0110 | norm_sim=0.1574
  Header: 'גרושים' | raw_sim=-0.0149 | norm_sim=0.1125
  Header: 'נשואים' | raw_sim=-0.0247 | norm_sim=0.0000
  Header: 'פרודים' | raw_sim=0.0582 | norm_sim=0.9488
  Header: 'שני ההורים אינם בחיים' | raw_sim=0.0482 | norm_sim=0.8349
  Header: 'אלמן/ה' | raw_sim=0.0626 | norm_sim=1.0000
  Header: 'מצב משפחתי' | raw_sim=-0.0048 | norm_sim=0.2274
  Header: 'עמודה1' | raw_sim=0.009

In [49]:
import os
import json
import numpy as np
from real_embeddings import RealEmbeddingGenerator

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def semantic_alignment_report(table_names_json_path, headers_json_path, threshold=0.7):
    # Load JSONs (same as before)
     # Load JSONs
    with open(table_names_json_path, 'r', encoding='utf-8') as f:
        table_names = json.load(f)

    with open(headers_json_path, 'r', encoding='utf-8') as f:
        table_headers = json.load(f)

    # Initialize embedding generator

    report = {}


    # Instead of embedder = RealEmbeddingGenerator()
    import anthropic
    client = anthropic.Anthropic(api_key="")

    for header_key, header_texts in table_headers.items():
        # Make one API call per header
        prompt = f"""Given these table features: {header_texts}

        Match them to the most relevant table names from this list:
        {json.dumps(table_names, ensure_ascii=False)}

        Return JSON with format:
        {{"matches": [
            {{"table_key": "...", "table_name": "...", "confidence": 0.0-1.0}}
        ]}}
        ⚠️ IMPORTANT: Do not include any explanation, commentary, or extra text.
        Return strictly valid JSON in the format above.
        Only include matches with confidence > {threshold}.
        """

        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )
        content_text = response.content[0].text.strip()

        # Safe parsing
        try:
            matches = json.loads(content_text)
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse JSON for {header_key}. Raw output:\n{content_text}")
            matches = {"matches": []}

        report[header_key] = matches.get("matches", [])

    return report

import os
import json
import anthropic

def refine_with_first_round(initial_report, table_names_json_path, headers_json_path, threshold=0.4, api_key="<YOUR_API_KEY>"):
    """
    Second-pass alignment using the first-pass output.
    Provides the model all headers, table names, and previous matches,
    asking it to decide which matches are correct or need adjustment.
    """
    # Load JSONs
    with open(table_names_json_path, 'r', encoding='utf-8') as f:
        table_names = json.load(f)

    with open(headers_json_path, 'r', encoding='utf-8') as f:
        table_headers = json.load(f)

    client = anthropic.Anthropic(api_key=api_key)
    refined_report = {}

    for header_key, header_texts in table_headers.items():
        previous_matches = initial_report.get(header_key, [])

        prompt = f"""
        You are helping to match table headers to table names.

        Headers: {header_texts}

        Available table names:
        {json.dumps(table_names, ensure_ascii=False)}

        Previous first-round matches with confidence scores:
        {json.dumps(previous_matches, ensure_ascii=False)}

        For each header, decide which table(s) it should match.
        - Keep high-confidence matches.
        - Correct any matches if needed.
        - You may update confidence scores (0.0-1.0).
        - Only include matches with confidence > {threshold}.

        Return strictly valid JSON in this format:
        {{
          "matches": [
            {{"table_key": "...", "table_name": "...", "confidence": 0.0-1.0}}
          ]
        }}
        ⚠️ IMPORTANT: Do not include any explanation, commentary, or extra text.
                Return strictly valid JSON in the format above.
        """

        # API call
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )

        content_text = response.content[0].text.strip()

        try:
            new_matches = json.loads(content_text)
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse JSON for {header_key}. Raw output:\n{content_text}")
            new_matches = {"matches": []}

        refined_report[header_key] = new_matches.get("matches", [])

    return refined_report

# # Example usage:
chapter_report = semantic_alignment_report(
    "/content/excel_and_json_2019_2021_2024/2019/4/4_2019.json",
    "/content/excel_and_json_2019_2021_2024/2019/4/4_2019_headers.json",
    threshold=0.4
)
# Example usage:
refined_report = refine_with_first_round(chapter_report,
                                         "/content/excel_and_json_2019_2021_2024/2019/4/4_2019.json",
                                         "/content/excel_and_json_2019_2021_2024/2019/4/4_2019_headers.json",
                                         threshold=0.4,
                                         api_key="")
print(json.dumps(refined_report, ensure_ascii=False, indent=2))



{
  "1_4_2019": [
    {
      "table_key": "6_4_2019",
      "table_name": "תרשים 4ו': שיעור הילדים שהושמו בידי משרד הרווחה בפנימיות לילדים בסיכון, \nלפי מצב משפחתי של ההורים / תשע\"ט (2018/19)",
      "confidence": 0.8
    },
    {
      "table_key": "7_4_2019",
      "table_name": "תרשים 4ז': התפלגות הסמנים* של ילדים שהושמו בפנימיות לילדים בסיכון, לפי מגדר / תשע\"ט (2018/19)",
      "confidence": 0.7
    },
    {
      "table_key": "5_4_2019",
      "table_name": "תרשים 4ה': התפלגות גילי הילדים בפנימיות לילדים בסיכון / תשע\"ט (2018/19)",
      "confidence": 0.6
    },
    {
      "table_key": "2_4_2019",
      "table_name": "תרשים 4ב': מספר הילדים שהושמו בידי משרד הרווחה במסגרות חוץ- ביתיות / תשע\"ד (2013/2014) – תשע\"ח (2017/2018)",
      "confidence": 0.5
    },
    {
      "table_key": "1_4_2019",
      "table_name": "תרשים 4א': התפלגות סוג משפחה של ילדים* בפנימיות של המינהל לחינוך התיישבותי פנימייתי ועליית הנוער / תשע\"ח (2017/18)",
      "confidence": 0.5
    }
  ],
  "4_4_2019"